# Chatbot

This notebook uses the same direct Ollama `/api/chat` approach as `../scripts/chat.py`, but presents it as a Jupyter chat panel. It can run as a local text chatbot and optionally speak replies through `sdk_client.Robot`.


In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import notebook UI helpers and the standard-library HTTP client used by the Ollama script.


In [2]:
import json
import string
import threading
import time
import urllib.error
import urllib.request
from difflib import SequenceMatcher

import ipywidgets as widgets
from IPython.display import display

from sdk_client import Robot

try:
    import rclpy
    from rclpy.node import Node
    from rclpy.executors import SingleThreadedExecutor
    from std_msgs.msg import String
    MIC_IMPORT_ERROR = None
except Exception as exc:
    rclpy = None
    Node = object
    SingleThreadedExecutor = None
    String = None
    MIC_IMPORT_ERROR = exc


DEFAULT_SYSTEM_PROMPT = (
    "You are the voice of a Unitree humanoid robot. Chat naturally with nearby people. "
    "Reply in no more than 25 words. "
    "Do not mention that you are a language model. Do not use markdown or hidden reasoning."
)


def clean_reply(text):
    text = str(text).strip()
    while "<think>" in text and "</think>" in text:
        before, rest = text.split("<think>", 1)
        _hidden, after = rest.split("</think>", 1)
        text = (before + after).strip()
    return " ".join(text.split())


FILLER_TEXTS = {"ah", "eh", "er", "hmm", "hm", "mm", "uh", "um", "嗯", "呃", "啊"}


def decode_payload(raw):
    try:
        payload = json.loads(str(raw))
        if isinstance(payload, dict):
            return payload
    except json.JSONDecodeError:
        pass
    return {"raw": str(raw), "text": str(raw)}


def payload_index(payload):
    try:
        value = payload.get("index")
        return int(value) if value is not None else None
    except Exception:
        return None


def similar(left, right):
    return SequenceMatcher(None, str(left).lower().strip(), str(right).lower().strip()).ratio()


def is_filler(text):
    normalized = str(text).strip().lower().strip(string.punctuation + "，。！？、；：")
    return normalized in FILLER_TEXTS


Configure Ollama and build the chat state. These defaults mirror `../scripts/chat.py`; override them with environment variables before running the cell.


In [3]:
def ollama_tags(base_url, timeout=1.5):
    request = urllib.request.Request(f"{str(base_url).rstrip('/')}/api/tags", method="GET")
    with urllib.request.urlopen(request, timeout=float(timeout)) as response:
        return json.loads(response.read().decode("utf-8"))


def choose_ollama_endpoint():
    configured = os.environ.get("G1_OLLAMA_URL") or os.environ.get("OLLAMA_HOST")
    candidates = []
    if configured:
        candidates.append(configured)
    candidates.extend([
        "http://127.0.0.1:11434",
        "http://192.168.123.164:11434",
        "http://localhost:11434",
    ])
    seen = set()
    errors = []
    for candidate in candidates:
        base = str(candidate).rstrip("/")
        if base in seen:
            continue
        seen.add(base)
        try:
            tags = ollama_tags(base)
            return base, tags
        except Exception as exc:
            errors.append(f"{base}: {exc}")
    raise RuntimeError("Could not reach Ollama. Tried " + "; ".join(errors))


def choose_model(tags):
    available = [m.get("name") or m.get("model") for m in tags.get("models", [])]
    available = [m for m in available if m]
    requested = os.environ.get("G1_CHAT_MODEL")
    if requested:
        if requested not in available:
            print(f"Requested G1_CHAT_MODEL={requested!r} is not installed. Available: {available}")
        return requested
    for preferred in ("qwen3.5:9b", "granite4.1:3b", "qwen3-vl:8b"):
        if preferred in available:
            return preferred
    if available:
        return available[0]
    return "granite4.1:3b"


OLLAMA_URL, OLLAMA_TAGS = choose_ollama_endpoint()
MODEL = choose_model(OLLAMA_TAGS)
SYSTEM_PROMPT = os.environ.get("G1_CHAT_SYSTEM", DEFAULT_SYSTEM_PROMPT)
TEMPERATURE = float(os.environ.get("G1_CHAT_TEMPERATURE", "0.4"))
TIMEOUT_S = float(os.environ.get("G1_CHAT_TIMEOUT", "30"))
MAX_HISTORY = int(os.environ.get("G1_CHAT_MAX_HISTORY", "4"))
NUM_PREDICT = int(os.environ.get("G1_CHAT_NUM_PREDICT", "48"))
NUM_CTX = int(os.environ.get("G1_CHAT_NUM_CTX", "1024"))
KEEP_ALIVE = os.environ.get("G1_CHAT_KEEP_ALIVE", "15m")
NUM_THREAD = os.environ.get("G1_CHAT_NUM_THREAD")
MIC_TOPIC = os.environ.get("G1_CHAT_MIC_TOPIC", "/audio_msg,/audio_msg/filter")
MIC_MIN_CONFIDENCE = float(os.environ.get("G1_CHAT_MIC_MIN_CONFIDENCE", "0.0"))
POST_SPEAK_IGNORE_S = float(os.environ.get("G1_CHAT_POST_SPEAK_IGNORE_S", "1.5"))
ANSWER_FILLERS = os.environ.get("G1_CHAT_ANSWER_FILLERS", "0") == "1"

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
robot = None
speak_replies = False
mic_bridge = None
mic_executor = None
last_audio_index = None
last_audio_text = None
last_reply_text = None
last_reply_ts = 0.0


def post_ollama_chat(body, timeout=TIMEOUT_S):
    data = json.dumps(body).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/chat",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=float(timeout)) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc


def ask_ollama(user_text):
    history = messages + [{"role": "user", "content": str(user_text)}]
    del history[1:max(1, len(history) - max(1, MAX_HISTORY))]
    options = {
        "temperature": float(TEMPERATURE),
        "num_predict": int(NUM_PREDICT),
        "num_ctx": int(NUM_CTX),
    }
    if NUM_THREAD:
        options["num_thread"] = int(NUM_THREAD)
    body = {
        "model": MODEL,
        "messages": history,
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "think": False,
        "options": options,
    }
    started = time.time()
    result = post_ollama_chat(body)
    reply = clean_reply(result.get("message", {}).get("content", ""))
    if not reply:
        reply = "I heard you, but I am not sure how to answer that yet."
    history.append({"role": "assistant", "content": reply})
    messages[:] = history
    return reply, time.time() - started


def warm_up_ollama():
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "Answer with one short word."},
            {"role": "user", "content": "Ready?"},
        ],
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "think": False,
        "options": {"temperature": 0, "num_predict": 2, "num_ctx": int(NUM_CTX)},
    }
    post_ollama_chat(body, timeout=TIMEOUT_S)

print(f"Ollama chat ready: url={OLLAMA_URL} model={MODEL} available={[m.get('name') for m in OLLAMA_TAGS.get('models', [])]}")


Ollama chat ready: url=http://192.168.123.164:11434 model=granite4.1:3b available=['qwen3-vl:8b', 'granite4.1:3b']


Optional robot speech binding. Set `enable_robot_speech = True` before running this cell if replies should be spoken.


In [4]:
enable_robot_speech = False

if enable_robot_speech:
    robot = Robot(iface=IFACE, domain_id=DOMAIN_ID, safety_boot=False, auto_start_sensors=False)
    speak_replies = True
    print("Robot speech enabled.")
else:
    print("Robot speech disabled. Set enable_robot_speech=True and rerun this cell to speak replies.")


Robot speech disabled. Set enable_robot_speech=True and rerun this cell to speak replies.


Run the chat panel. Typed prompts and microphone ASR prompts both use the same Ollama request path. The microphone subscriber listens to `/audio_msg` by default.


In [5]:
prompt = widgets.Textarea(placeholder="Type a message...", layout=widgets.Layout(width="100%", height="90px"))
send = widgets.Button(description="Send", button_style="success")
warmup = widgets.Button(description="Warm Up")
clear = widgets.Button(description="Clear")
speak = widgets.Checkbox(value=speak_replies, description="speak replies")
mic_topic = widgets.Text(value=MIC_TOPIC, description="Mic topic", layout=widgets.Layout(width="320px"))
min_conf = widgets.FloatSlider(value=MIC_MIN_CONFIDENCE, min=0.0, max=1.0, step=0.05, description="Min conf")
answer_fillers = widgets.Checkbox(value=ANSWER_FILLERS, description="answer fillers")
start_mic = widgets.Button(description="Start Mic", button_style="info")
stop_mic = widgets.Button(description="Stop Mic")
chat_log = widgets.Textarea(layout=widgets.Layout(width="100%", height="420px"), disabled=True)


def add(line):
    chat_log.value = (chat_log.value + line + "\n")[-12000:]


def speak_reply(reply):
    global robot, last_reply_text, last_reply_ts
    last_reply_text = reply
    last_reply_ts = time.time()
    if not speak.value:
        return
    try:
        if robot is None:
            robot = Robot(iface=IFACE, domain_id=DOMAIN_ID, safety_boot=False, auto_start_sensors=False)
            add("[speech] Robot audio client ready")
        code = robot.say(reply)
        add(f"[speech] robot.say rc={code}")
        last_reply_ts = time.time()
    except Exception as exc:
        add(f"error> speech failed: {exc}")


def handle_user_text(text, source="you"):
    text = str(text).strip()
    if not text:
        return
    add(f"{source}> {text}")
    try:
        reply, elapsed = ask_ollama(text)
        add(f"bot> {reply}")
        add(f"[ollama] {elapsed:.1f}s")
        speak_reply(reply)
    except Exception as exc:
        add(f"error> {exc}")


def should_answer_audio(text, confidence, index):
    global last_audio_index, last_audio_text, last_reply_text, last_reply_ts
    now = time.time()
    if not text or float(confidence or 0.0) < float(min_conf.value):
        return False
    if not answer_fillers.value and is_filler(text):
        return False
    if not any(char.isalnum() for char in text):
        return False
    if index is not None and index == last_audio_index:
        return False
    if index is None and text == last_audio_text and (now - last_reply_ts) < 2.0:
        return False
    if (now - last_reply_ts) < float(POST_SPEAK_IGNORE_S):
        return False
    if last_reply_text and similar(text, last_reply_text) >= 0.82:
        return False
    last_audio_index = index
    last_audio_text = text
    return True


class NotebookMicBridge(Node):
    def __init__(self, topics):
        super().__init__("notebook_robot_chat")
        raw_topics = str(topics).replace(";", ",").split(",")
        self.topics = [t.strip() for t in raw_topics if t.strip()] or ["/audio_msg"]
        self.topic = ",".join(self.topics)
        self.subscriptions = [self.create_subscription(String, topic, self.on_audio_msg, 10) for topic in self.topics]

    def on_audio_msg(self, msg):
        raw = str(msg.data)
        payload = decode_payload(raw)
        text = str(payload.get("text") or payload.get("raw") or "").strip()
        confidence = float(payload.get("confidence", 0.0) or 0.0)
        index = payload_index(payload)
        add(f"[mic raw] text={text!r} confidence={confidence:.2f} index={index} payload={raw[:180]}")
        if should_answer_audio(text, confidence, index):
            handle_user_text(text, source="mic")


def on_send(_):
    text = prompt.value.strip()
    if not text:
        return
    prompt.value = ""
    handle_user_text(text, source="you")


def on_warmup(_):
    try:
        started = time.time()
        warm_up_ollama()
        add(f"[ollama] warm-up finished in {time.time() - started:.1f}s")
    except Exception as exc:
        add(f"error> warm-up failed: {exc}")


def on_clear(_):
    messages[:] = [{"role": "system", "content": SYSTEM_PROMPT}]
    chat_log.value = ""


def on_start_mic(_):
    global mic_bridge, mic_executor
    if MIC_IMPORT_ERROR is not None:
        add(f"error> microphone unavailable: {MIC_IMPORT_ERROR}")
        return
    if mic_bridge is not None:
        add(f"[mic] already listening on {mic_bridge.topic}")
        return
    try:
        if not rclpy.ok():
            rclpy.init(args=None)
        mic_bridge = NotebookMicBridge(mic_topic.value.strip() or "/audio_msg,/audio_msg/filter")
        mic_executor = SingleThreadedExecutor()
        mic_executor.add_node(mic_bridge)
        thread = threading.Thread(target=mic_executor.spin, daemon=True)
        thread.start()
        mic_bridge._spin_thread = thread
        add(f"[mic] listening on {mic_bridge.topic}; speak after this line to see [mic raw] entries")
    except Exception as exc:
        try:
            if mic_bridge is not None:
                mic_bridge.destroy_node()
            if mic_executor is not None:
                mic_executor.shutdown()
        finally:
            mic_bridge = None
            mic_executor = None
        add(f"error> start mic failed: {exc}")


def on_stop_mic(_):
    global mic_bridge, mic_executor
    if mic_bridge is None and mic_executor is None:
        add("[mic] not running")
        return
    try:
        node = mic_bridge
        executor = mic_executor
        mic_bridge = None
        mic_executor = None
        if executor is not None:
            executor.shutdown()
        if node is not None:
            node.destroy_node()
        thread = getattr(node, "_spin_thread", None) if node is not None else None
        if thread is not None and thread.is_alive():
            thread.join(timeout=1.0)
        add("[mic] stopped")
    except Exception as exc:
        add(f"error> stop mic failed: {exc}")

send.on_click(on_send)
warmup.on_click(on_warmup)
clear.on_click(on_clear)
start_mic.on_click(on_start_mic)
stop_mic.on_click(on_stop_mic)
display(widgets.VBox([
    prompt,
    widgets.HBox([send, warmup, clear, speak]),
    widgets.HBox([mic_topic, min_conf, answer_fillers, start_mic, stop_mic]),
    chat_log,
]))
